<a href="https://colab.research.google.com/github/mojordan235-oss/Project/blob/main/CMP_414_Artifical_Intellengce_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Hyperparameters for attention models
n_embd = 64 # Affects the size of the bigram model
n_head = 4
# head_size = 16
n_layer = 8
device = "gpu"
max_int=5000
evalinterval=100
eval_iters=200
batch_size=16


! wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
block_size = 32

dropout = 0.25 # The number determines the proportion of removed neurons
#imporve the attention layers
# Redefine the FeedForward block with dropout
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4* n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

--2026-05-17 23:05:38--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-05-17 23:05:38 (21.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [ ]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [ ]:
@torch.no_grad() # A function modifier to improve efficiency
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# Redefine MultiHeadAttention with dropout
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))      # ResNet structure is used
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class SelfAttentionModel4(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
# A self-attention block
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)     # the location information
        self.query = nn.Linear(n_embd, head_size, bias=False)   # the expected token
        self.value = nn.Linear(n_embd, head_size, bias=False)  # the meanings of tokens
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        # Generate the key, query, and value matrices
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        v = self.value(x) # (B,T,C)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

In [ ]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [ ]:
from datetime import datetime
import time
model=SelfAttentionModel4()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m=model.to(device)

max_int=5000
evalinterval=100

learning_rate=1e-3

optimizer=torch.optim.AdamW(model.parameters(), lr=learning_rate)

start=datetime.now()
print("Training Start:",start)


for i in range(100,max_int):


  if i %  evalinterval == 0 or i == max_int-1:
    loss=estimate_loss()
    print(f"step {i}:train loss {loss['train']:.4f},val loss {loss['val']:.4f}")

    xb,yb= get_batch('train')

    logits,loss1= model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss1.backward()
    optimizer.step()


end=datetime.now()

print("Training End:",start)

elapsed=end-start

print("Time Elapsed:",elapsed)



Training Start: 2026-05-17 23:05:46.355351
step 100:train loss 4.3325,val loss 4.3263
step 200:train loss 3.9890,val loss 3.9886
step 300:train loss 3.7675,val loss 3.7746
step 400:train loss 3.6387,val loss 3.6520
step 500:train loss 3.5762,val loss 3.5838
step 600:train loss 3.5314,val loss 3.5397
step 700:train loss 3.4830,val loss 3.5025
step 800:train loss 3.4543,val loss 3.4841
step 900:train loss 3.4295,val loss 3.4524
step 1000:train loss 3.4123,val loss 3.4341
step 1100:train loss 3.3964,val loss 3.4161
step 1200:train loss 3.3745,val loss 3.4036
step 1300:train loss 3.3545,val loss 3.3824
step 1400:train loss 3.3356,val loss 3.3730
step 1500:train loss 3.3317,val loss 3.3534
step 1600:train loss 3.3120,val loss 3.3391
step 1700:train loss 3.3020,val loss 3.3308
step 1800:train loss 3.2948,val loss 3.3195
step 1900:train loss 3.2775,val loss 3.3249
step 2000:train loss 3.2650,val loss 3.2985
step 2100:train loss 3.2560,val loss 3.2927
step 2200:train loss 3.2477,val loss 3.287

In [ ]:
text=torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(text,max_new_tokens=5000)[0].tolist()))


Ced Iax
Srovd ndw
Q
sYIa3,s gdend witer I Ie, wovkesiwat,
J hf; xoThRe h, w
ngeUa f cousO a  a.cedndowe TE. ho yat tamat
eneuyQinve Voenan.r ;, 3 Sber nenemo'n ikum d UItantith' at  aOsen ornnqoi?onan mine inocedae noo ps memaouny huthe the s Yg are irte-ikicoHor You dhiIelerekeno wis? , ms Qot mon af !hathTisesgtyoid,n-rkNefeleZ,y s weNop :!esTo iBal, ls ssit,!bE, t Men :
ol thatoayC sed,inity
Z&Hke coIGC itnennt cou,, yl ciLo'od a aBitsQnak eramirdere eGou cncLdj Hery and souQudhq, Snmidty e te orat f s thf y fowh tIW g t
EF
fqy:N onchoE athsoeouads aS !h,J, fat him Tut heGacitrS-doGine gWl  b,?
;rr fVipeAeHvZ thig weu yo
CW tt atsO;reooBcoMoWSs nta, hen hg E
jere I,ot ing, hid ,
i3 z t
uoo y gohamouechenfiamere
;ay
thp d. QwN, t YiSceserX h ntaBl bosvee nging y
wouto,  ?Lowoom nIheVr odr,ia?ediard.onzR: ofthelemeanof!as wsat le s skitullYciNJ
C hres 
LsrhATu,ig G dUto weSe g KO?r bkefqi q he,ed geab.is ewouhaHatKe d 3wonm
T:u,:tasinvd uinc!unesA senty th w idu tei d t meRy AifackDf

In [ ]:
# My analyze in the project is that for the mdodel for to train takes an long to finish I wish it was little more faster for the model to be trained.